# LLM Evaluation on Greek Protipa Exams

This notebook evaluates multiple LLMs on the `PennyK98/protipa_exams_dataset` from Hugging Face. It focuses on Greek Language and Mathematics multiple-choice questions.

In [21]:
import json
import logging
import requests
import os
import sys
import random
import time
import traceback
from pathlib import Path

import lm_eval
import pandas as pd
import yaml
from datasets import load_dataset
from datasets import load_dataset, concatenate_datasets
from dotenv import load_dotenv
from lm_eval.models.openai_completions import OpenAIChatCompletion
from lm_eval.tasks import ConfigurableTask, TaskManager

import IPython.display

# Setup Logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

## 1. Environment Setup

In [22]:
load_dotenv()

# API Config
os.environ["OPENAI_API_KEY"] = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
os.environ["OPENAI_API_BASE"] = os.getenv("LITELLM_HOST")
api_base = os.getenv("LITELLM_HOST")

# Output Config
results_dir = Path(os.getenv("RESULTS_DIR", "tmp"))
results_dir.mkdir(parents=True, exist_ok=True)

models_to_test = ["gemma3-27b-it", "krikri-dpo-context"]
logger.info(f"Target models: {models_to_test}")

2026-01-16 11:27:20 - INFO - Target models: ['gemma3-27b-it', 'krikri-dpo-context']


In [23]:
# Ρυθμίσεις για να βρούμε τον κώδικα στο src
project_root = Path.cwd().parent 
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    logger.info(f"✅ Προστέθηκε το {src_path} στο path!")

try:
    from protipa_exams_dataset.data_loader import load_protipa_dataset, apply_matching_processing
    logger.info("🚀 Επιτυχία! Το data_loader φορτώθηκε από το src.")
except ImportError as e:
    logger.warning(f"⚠️ Δεν βρέθηκε η συνάρτηση/module. Έλεγξε τα ονόματα στο src. Error: {e}")

2026-01-16 11:27:22 - INFO - 🚀 Επιτυχία! Το data_loader φορτώθηκε από το src.


## 2. Load and Prepare Dataset

Παράδειγμα

In [ ]:
#logger.info("Loading dataset...")
#original_dataset = load_dataset("PennyK98/protipa_exams_dataset", split="test")

#Ελέγχουμε μόνο τα Μαθηματικά και τη Γλώσσα και μόνο ερωτήσεις τύπου Multiple Choice χωρίς εικόνες
#def filter_dataset(dataset):
    #filtered = []
    #subjects = ['ΓΛΩΣΣΑ', 'ΜΑΘΗΜΑΤΙΚΑ']
    #for item in dataset:
        #multimodal_status = item.get('multimodality')
        #if (item['subject'] in subjects and 
            #item['exercise_type'] == 'Multiple Choice' and 
            #',' not in str(item['answer_index']) and multimodal_status == 'no'):
            #filtered.append(item)
    #return filtered

#all_filtered = filter_dataset(original_dataset)
#logger.info(f"Filtered dataset size: {len(all_filtered)}")

# Sample for pilot test
#pilot_100 = random.sample(all_filtered, min(len(all_filtered), 100))

# Save temporary JSON for lm_eval ingestion
#json_path = (results_dir / "pilot_data_test.json").resolve()
#with open(json_path, 'w', encoding='utf-8') as f:
    #json.dump(pilot_100, f, ensure_ascii=False, indent=4)

#logger.info(f"Pilot data (100 samples) saved to: {json_path}")

2026-01-09 09:36:01 - INFO - Loading dataset...
2026-01-09 09:36:05 - INFO - Filtered dataset size: 192
2026-01-09 09:36:05 - INFO - Pilot data (100 samples) saved to: C:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πρότυπα_Πειραματικά\tmp\pilot_data_test.json


In [24]:
#Script για εύκολη χρήση των νέων συναρτήσεων από το data_loader.py
from protipa_exams_dataset.data_loader import (
    load_protipa_dataset, 
    filter_dataset, 
    apply_matching_processing, 
)

# Ρύθμιση φακέλου αποτελεσμάτων
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

logger.info("🚀 Ξεκινάει η διαδικασία με τις νέες συναρτήσεις...")

# --- Βήμα 1: Φόρτωση από Hugging Face ---
# Καλεί την έτοιμη συνάρτηση που κατεβάζει και ενώνει train + test
hf_dataset = load_protipa_dataset() 
logger.info(f"✅ Loaded raw dataset from HF. Size: {len(hf_dataset)}")

# --- Βήμα 2: Φιλτράρισμα ---
# Καλεί την έτοιμη συνάρτηση φιλτραρίσματος (Γλώσσα, Μαθηματικά κλπ.)
# Προσοχή: Επιστρέφει ΛΙΣΤΑ, όχι DataFrame ακόμα.
filtered_list = filter_dataset(hf_dataset)
logger.info(f"✅ Filtered items: {len(filtered_list)}")

# --- Βήμα 3: Μετατροπή σε DataFrame ---
# Απαραίτητο για να δουλέψουν οι επόμενες συναρτήσεις (fix matching & clean paths)
df = pd.DataFrame(filtered_list)

# --- Βήμα 3b: ΔΙΟΡΘΩΣΗ ΤΩΝ ΕΙΚΟΝΩΝ ---
# Επειδή το HF μας δίνει PIL Objects (εικόνες) και εμείς θέλουμε Strings (ονόματα) για το JSON:
def extract_filename_from_object(val):
    # Αν είναι ήδη string (κείμενο), το κρατάμε
    if isinstance(val, str):
        return os.path.basename(val.replace('\\', '/'))
    # Αν είναι λίστα, το εφαρμόζουμε σε κάθε στοιχείο
    if isinstance(val, list):
        return [extract_filename_from_object(x) for x in val]
    # Αν είναι PIL Image (το PngImageFile που έβγαλε το error), παίρνουμε το filename του
    if hasattr(val, 'filename') and val.filename:
        return os.path.basename(val.filename.replace('\\', '/'))
    
    return None

logger.info("🖼️ Converting Image Objects to Filenames for JSON...")
if 'image' in df.columns:
    df['image'] = df['image'].apply(extract_filename_from_object)

# --- Βήμα 4: Διόρθωση Matching & Καθαρισμός ---
logger.info("🔄 Applying Matching Fix & Cleaning Paths...")
df = apply_matching_processing(df)  # Διορθώνει τα Matching

# --- Βήμα 5: Αποθήκευση ---
full_data_path = results_dir / "full_dataset_for_eval.json"
df.to_json(full_data_path, orient="records", force_ascii=False, indent=4)

logger.info(f"   Questions ready for evaluation: {len(df)}")

2026-01-16 11:27:29 - INFO - 🚀 Ξεκινάει η διαδικασία με τις νέες συναρτήσεις...
2026-01-16 11:27:29 - INFO - Loading dataset from Hugging Face: PennyK98/protipa_exams_dataset
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 7ff7cb2b-e53c-480a-b6da-509362b13f5d)')' thrown while requesting HEAD https://huggingface.co/datasets/PennyK98/protipa_exams_dataset/resolve/main/README.md
2026-01-16 11:27:39 - WARNING - '(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 7ff7cb2b-e53c-480a-b6da-509362b13f5d)')' thrown while requesting HEAD https://huggingface.co/datasets/PennyK98/protipa_exams_dataset/resolve/main/README.md
Retrying in 1s [Retry 1/5].
2026-01-16 11:27:39 - WARNING - Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: a918f918-e29f-4d08

## 3. Define Evaluation Task Template

Task_config για ερωτήσεις κλειστού τύπου

In [25]:
path_str = str(full_data_path)
logger.info(f"Setting up task with data file: {path_str}")

task_config = {
    "task": "greek_protipa_exams",
    "dataset_path": "json",
    "num_fewshot": 0,      
    "dataset_kwargs": {
            "data_files": path_str
        },
    "test_split": "train",
    "output_type": "generate_until",
    "doc_to_text": (
        "{% if input %}{{input}}\n{% endif %}"
        "Ερώτηση: {{question}}\n"
        "Επιλογές:\n"
        "{% for choice in choices %}"
        "{{loop.index0}}. {{choice}}\n"
        "{% endfor %}\n"
        "### ΟΔΗΓΙΑ ΜΟΡΦΟΠΟΙΗΣΗΣ (CRITICAL)\n"
        "Πρέπει να παρέχεις ΜΟΝΟ τον αριθμό του δείκτη (index) της σωστής επιλογής (π.χ. 0, 1, 2, 3...).\n"
        "ΠΡΟΣΟΧΗ: Ο αριθμός '2' που χρησιμοποιείται στα παραδείγματα παρακάτω είναι ΤΥΧΑΙΟΣ και αφορά μόνο τη ΜΟΡΦΗ της απάντησης εδώ.\n"
        "Η σωστή απάντηση εξαρτάται αποκλειστικά από την ερώτηση και μπορεί να είναι οποιοσδήποτε αριθμός.\n"
        "Μην επεξηγείς και μην γράφεις ολόκληρες προτάσεις.\n\n"
        "Παραδείγματα Μορφής:\n"
        "❌ ΛΑΘΟΣ: \"Η σωστή επιλογή είναι η 2.\"\n"
        "❌ ΛΑΘΟΣ: \"(2)\"\n"
        "✅ ΣΩΣΤΟ: 2 (ή 0 ή 1 ή 3... ανάλογα με τη σωστή επιλογή)\n\n"
        "Απάντηση: "
    ),
    "doc_to_target": "{{ (answer_index | string).split(',')[0] }}",
    "generation_kwargs": {
        "until": ["\n"],
        "max_gen_toks": 50,
        "do_sample": False,
        "temperature": 0.0 
    },
    "filter_list": [
        {
            "name": "strict-match",
            "filter": [
                {"function": "regex", "regex_pattern": r"(?<![0-9/])([0-9])(?![0-9/])"},
                {"function": "take_first"}
            ]
        }
    ],
    "metric_list": [
        {"metric": "exact_match", "aggregation": "mean", "higher_is_better": True}
    ]
}

# Workaround for Task Config loading
task_dir = results_dir
task_dir.mkdir(parents=True, exist_ok=True)
with open(task_dir / "greek_protipa.yaml", "w", encoding='utf-8') as f:
    yaml.dump(task_config, f, allow_unicode=True)

custom_task = ConfigurableTask(config=task_config)
task_dict = {"greek_protipa_exams": custom_task}
logger.info("Evaluation task defined successfully.")

2026-01-16 11:28:03 - INFO - Setting up task with data file: c:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πρότυπα_Πειραματικά\results\full_dataset_for_eval.json


Generating train split: 0 examples [00:00, ? examples/s]

2026-01-16 11:28:04 - INFO - Evaluation task defined successfully.


## 4. Run Evaluation

In [26]:
from protipa_exams_dataset.evaluation import run_evaluation

comparison_results = {}
all_samples = {}

EVAL_LIMIT = None  # Adjust this to run more/less samples

for model_name in models_to_test:
    
    results = run_evaluation(
        model_name=model_name,
        api_base=api_base,
        task_dict=task_dict,
        eval_limit=EVAL_LIMIT
    )

    if results is None:
        continue

    try:
        scores = results['results']['greek_protipa_exams']
        comparison_results[model_name] = scores
        
        if 'samples' in results and 'greek_protipa_exams' in results['samples']:
            samples = results['samples']['greek_protipa_exams']
            for i, s in enumerate(samples):
                if i not in all_samples:
                    doc = s.get('doc', {})
                    all_samples[i] = {
                        "Question": doc.get('question', 'N/A'),
                        "Subject": doc.get('subject', 'N/A'),
                        "Exercise Type": doc.get('exercise_type', 'N/A'),
                        "Year": doc.get('year', 'N/A'),
                        "Level": doc.get('school_level', 'N/A'),
                        "Ground Truth": str(doc.get('answer_index', 'N/A')),
                        "Choices": " | ".join(doc.get('choices', [])),
                        "Multimodality": doc.get('multimodality', 'no'),
                        "Image Path": str(doc.get('image', '')),
                        "Model Predictions": {},
                        "Raw Responses": {}
                    }
                
                all_samples[i]["Model Predictions"][model_name] = s.get('filtered_resps', ["N/A"])[0]
                all_samples[i]["Raw Responses"][model_name] = s.get('resps', [["N/A"]])[0][0]

        acc = scores.get('exact_match,strict-match', scores.get('acc', 0.0))
        logger.info(f"✅ Success! {model_name} Accuracy: {acc:.2%}")
        
        logger.info("⏳ Waiting 2 seconds before next model...")
        time.sleep(2)

    except Exception as e:
        logger.error(f"❌ Error processing results for {model_name}: {e}")
        logger.error(traceback.format_exc())

2026-01-16 11:28:10 - INFO - Starting evaluation for model: gemma3-27b-it
2026-01-16 11:28:10 - INFO - Using max length 2048 - 1


2026-01-16 11:28:10 - INFO - Concurrent requests are disabled. To enable concurrent requests, set `num_concurrent` > 1.
2026-01-16 11:28:10 - INFO - Using tokenizer None
2026-01-16 11:28:10 - WARNING - Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
2026-01-16 11:28:10 - INFO - Building contexts for greek_protipa_exams on rank 0...
100%|██████████| 1368/1368 [00:02<00:00, 676.01it/s]
2026-01-16 11:28:12 - INFO - Running generate_until requests
2026-01-16 11:28:12 - INFO - Tokenized requests are disabled. Context + generation length is not checked.
Requesting API: 100%|██████████| 1368/1368 [19:51<00:00,  1.15it/s]
2026-01-16 11:48:07 - INFO - ✅ Success! gemma3-27b-it Accuracy: 61.40%
2026-01-16 11:48:07 - INFO - ⏳ Waiting 2 seconds before next model...
2026-01-16 11:48:09 - INFO - Starting evaluation for model: krikri-dpo-context
2026-01-16 11:48:09 - INFO - Using max length 2048 - 1
2026-01-16 11:48:09 - IN

## 5. Results Table

In [27]:
if all_samples:
    table_data = []
    for idx, data in all_samples.items():
        row = {
            "ID": idx,
            "Subject": data["Subject"],
            "Exercise Type": data.get("Exercise Type", "N/A"),
            "Multimodality": data.get("Multimodality", "no"), 
            "Image Path": data.get("Image Path", ""),         
            "Year": data["Year"],
            "Level": data["Level"],
            "Question": data["Question"],
            "Choices": data["Choices"],
            "Ground Truth": data["Ground Truth"]
        }
        
        for m in models_to_test:
            # Αποθηκεύουμε την απάντηση του κάθε μοντέλου
            row[f"{m}_pred"] = data["Model Predictions"].get(m, "N/A")
            # row[f"{m}_raw"] = data["Raw Responses"].get(m, "N/A") 
            
        table_data.append(row)
    
    df_results = pd.DataFrame(table_data)
    
    model_names_str = "_".join([m.split("/")[-1] for m in models_to_test])
    filename = f"eval_results_{model_names_str}.csv"
    results_file = results_dir / filename

    # Αποθήκευση
    df_results.to_csv(results_file, index=False, encoding='utf-8-sig')
    logger.info(f"💾 Table saved successfully to: {results_file}")
    
    # Εμφάνιση των πρώτων 5 γραμμών για να θαυμάσεις το αποτέλεσμα!
    display(df_results.head())
else:
    logger.warning("⚠️ No samples collected to save!")

2026-01-16 12:01:20 - INFO - 💾 Table saved successfully to: c:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πρότυπα_Πειραματικά\results\eval_results_gemma3-27b-it_krikri-dpo-context.csv


,ID,Subject,Exercise Type,Multimodality,Image Path,Year,Level,Question,Choices,Ground Truth,gemma3-27b-it_pred,krikri-dpo-context_pred
0,0,ΓΛΩΣΣΑ,Multiple Choice,no,[],2020,ΛΥΚΕΙΟ,Να συμπληρώσετε σωστά τη φράση: Ο κινηματογραφ...,"Α. ό,τι συνέβη. | Β. ό,τι συνέβει. | Γ. ότι συ...",0,2,2
1,1,ΜΑΘΗΜΑΤΙΚΑ,Multiple Choice,no,[],2021,ΛΥΚΕΙΟ,Παντοπώλης είχε ένα γεμάτο τσουβάλι με ρύζι. Τ...,A. 9 | B. 12 | Γ. 18 | Δ. 21,2,2,2
2,2,ΜΑΘΗΜΑΤΙΚΑ,Multiple Choice,no,[],2021,ΛΥΚΕΙΟ,"Ποια είναι η διάμεσος των αριθμών: $\sqrt{3}, 8$;","A. 8,5 | B. 9 | Γ. 10 | Δ. 8",0,0,2
3,3,ΜΑΘΗΜΑΤΙΚΑ,Multiple Choice,no,[],2020,ΓΥΜΝΑΣΙΟ,Η παράσταση $\frac{2}{5}\cdot(4-\frac{3}{7})$ ...,A. $\frac{10}{7}$ | B. $\frac{1}{2}$ | Γ. 7 | ...,0,0,2
4,4,ΓΛΩΣΣΑ,True/False,no,[],2019,ΛΥΚΕΙΟ,"Σύμφωνα με το κείμενο Β, να χαρακτηρίσετε ως *...",α. Σωστό | β. Λάθος,1,0,0


## 6. Performance Summary

In [28]:
if comparison_results:
    summary_data = []
    for model, metrics in comparison_results.items():
        acc = metrics.get('exact_match,strict-match', metrics.get('acc', 0.0))
        
        summary_data.append({
            "Model": model, 
            "Accuracy": acc
        })
    
    df_summary = pd.DataFrame(summary_data)
    display (df_summary)
    
    # Αποθήκευση του πίνακα σε CSV
    #summary_file = results_dir / "evaluation_summary_scores.csv"
    #df_summary.to_csv(summary_file, index=False)
    #logger.info(f"📊 Summary saved to: {summary_file}")

    #print("\n=== ΤΕΛΙΚΗ ΒΑΘΜΟΛΟΓΙΑ ΜΟΝΤΕΛΩΝ ===")
    # Φτιάχνουμε ένα αντίγραφο για την εκτύπωση για να μην χαλάσουμε τα νούμερα στο dataframe
    #df_display = df_summary.copy()
    #df_display['Accuracy'] = df_display['Accuracy'].apply(lambda x: f"{x:.2%}")
    #display(df_display)

else:
    logger.warning("⚠️ No comparison results found to summarize.")

,Model,Accuracy
0,gemma3-27b-it,0.614035
1,krikri-dpo-context,0.389620


In [29]:
# Φορτώνουμε το CSV που μόλις φτιάξαμε
df = pd.read_csv(results_file)

print(f"📊 Φορτώθηκαν {len(df)} ερωτήσεις για ανάλυση.\n")

def normalize_val(val):
    """Καθαρίζει την τιμή για να γίνει σωστή σύγκριση."""
    s = str(val).strip()  # Αφαιρεί κενά και \n
    # Αν κατά λάθος έγινε 1.0 (float string), το κάνουμε 1
    if s.endswith(".0"):
        s = s[:-2]
    return s

# Υπολογισμός Σωστού/Λάθους δυναμικά για ΟΛΑ τα μοντέλα που έτρεξες
for model in models_to_test:
    col_name = f"{model}_pred"
    
    if col_name in df.columns:
        # Εφαρμόζουμε το καθάρισμα (normalize)
        gt_clean = df["Ground Truth"].apply(normalize_val)
        pred_clean = df[col_name].apply(normalize_val)
        
        # Συγκρίνουμε και φτιάχνουμε στήλη _correct (0 ή 1)
        df[f"{model}_correct"] = (gt_clean == pred_clean).astype(int)
        print(f"✅ Calculated accuracy column for: {model}")

#df.to_csv(results_file, index=False, encoding='utf-8-sig')

📊 Φορτώθηκαν 1368 ερωτήσεις για ανάλυση.

✅ Calculated accuracy column for: gemma3-27b-it
✅ Calculated accuracy column for: krikri-dpo-context


Ανάλυση RQ1: Ακρίβεια ανά μάθημα (κλειστού τύπου)

In [30]:
# Ανάλυση RQ1: Ακρίβεια ανά Είδος Άσκησης (Μάθημα)
print("\n" + "-" * 60)
print("🏆 RQ1: PERFORMANCE PER SUBJECT")
print("-" * 60)

model_cols = [c for c in df.columns if c.endswith('_correct')]

if not model_cols:
    logger.warning("⚠️ Δεν βρέθηκαν στήλες αποτελεσμάτων (_correct).")
else:
    #Υπολογίζουμε τη μέση τιμή (Accuracy) για κάθε μάθημα
    rq1_acc = df.groupby("Subject")[model_cols].mean()
    
    #Υπολογίζουμε το πλήθος των ερωτήσεων ανά μάθημα
    rq1_count = df.groupby("Subject")[model_cols[0]].count()
    
    rq1_final = pd.DataFrame()
    
    rq1_final['Total Questions'] = rq1_count
    
    for col in model_cols:
        clean_name = col.replace("_correct", "")
        rq1_final[clean_name] = (rq1_acc[col] * 100).round(1).astype(str) + '%'

    display(rq1_final)
    
    #rq1_csv_path = results_dir / "RQ1_performance_per_subject.csv"
    #rq1_final.to_csv(rq1_csv_path)
    #logger.info(f"💾 RQ1 Table saved to: {rq1_csv_path}")

logger.info("Evaluation analysis for RQ1 completed.")


------------------------------------------------------------
🏆 RQ1: PERFORMANCE PER SUBJECT
------------------------------------------------------------


,Total Questions,gemma3-27b-it,krikri-dpo-context
Subject,,,
ΓΛΩΣΣΑ,723,71.6%,45.4%
ΘΡΗΣΚΕΥΤΙΚΑ,90,77.8%,63.3%
ΜΑΘΗΜΑΤΙΚΑ,555,44.3%,26.3%


2026-01-16 12:01:36 - INFO - Evaluation analysis for RQ1 completed.


Ανάλυση RQ2: Απόδοση ανάλογα με το αν υπάρχει ή όχι εικόνα (κλειστού τύπου)

In [31]:
# Ανάλυση RQ2: Multimodality (Εικόνα vs Κείμενο)
# Ανάλυση RQ2: Multimodality (Visual Context)
print("\n" + "-" * 60)
print("🏆 RQ2: IMPACT OF MULTIMODALITY")
print("-" * 60)

# Βρίσκουμε τις στήλες των μοντέλων
model_cols = [c for c in df.columns if c.endswith('_correct')]

# Ομαδοποίηση
rq2_stats = df.groupby("Multimodality")[model_cols].mean()
rq2_count = df.groupby("Multimodality")[model_cols[0]].count()

# Μορφοποίηση
rq2_final = pd.DataFrame()
rq2_final["Total Questions"] = rq2_count

for col in model_cols:
    clean_name = col.replace("_correct", "")
    rq2_final[clean_name] = (rq2_stats[col] * 100).round(1).astype(str) + '%'

display(rq2_final)

# Αποθήκευση
#rq2_final.to_csv(results_dir / "RQ2_multimodality_impact.csv")


------------------------------------------------------------
🏆 RQ2: IMPACT OF MULTIMODALITY
------------------------------------------------------------


,Total Questions,gemma3-27b-it,krikri-dpo-context
Multimodality,,,
no,1176,63.5%,40.6%
yes,192,45.3%,28.1%


Ανάλυση RQ3: Ακρίβεια ανά είδος άσκησης (κλειστού τύπου)

In [32]:
# Ανάλυση RQ3: Ακρίβεια ανά Είδος Άσκησης
print("\n" + "-" * 60)
print("🏆 RQ3: PERFORMANCE PER CLOSED EXERCISE TYPE")
print("-" * 60)

model_cols = [c for c in df.columns if c.endswith('_correct')]

if not model_cols:
    logger.warning(f"⚠️ Δεν βρέθηκαν στήλες αποτελεσμάτων (_correct).")

else:
    rq3_acc = df.groupby("Exercise Type")[model_cols].mean()
    rq3_count = df.groupby("Exercise Type")[model_cols[0]].count()
    
    rq3_final = pd.DataFrame()
    rq3_final["Total Exercises"] = rq3_count
    
    for col in model_cols:
        clean_name = col.replace("_correct", "")
        
        rq3_final[clean_name] = (rq3_acc[col] * 100).round(1).astype(str) + '%'
    
    display(rq3_final)
    
    #rq3_csv_path = results_dir / "RQ3_performance_per_exercise_type.csv"
    #rq3_final.to_csv(rq3_csv_path)
    #logger.info(f"💾 RQ3 Table saved to: {rq3_csv_path}")

logger.info("Evaluation analysis for RQ3 completed.")


------------------------------------------------------------
🏆 RQ3: PERFORMANCE PER CLOSED EXERCISE TYPE
------------------------------------------------------------


,Total Exercises,gemma3-27b-it,krikri-dpo-context
Exercise Type,,,
Fill-in-the-gaps,9,88.9%,77.8%
Matching,4,0.0%,0.0%
Multiple Choice,1268,59.3%,37.1%
True/False,87,85.1%,60.9%


2026-01-16 12:02:32 - INFO - Evaluation analysis for RQ3 completed.


Διαθέσιμα μοντέλα

In [ ]:
api_key = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
api_base = os.getenv("LITELLM_HOST")

# Καθαρισμός URL
if api_base.endswith("/chat/completions"):
    api_base = api_base.replace("/chat/completions", "")
if not api_base.endswith("/v1"):
    api_base = api_base.rstrip("/") + "/v1"

try:
    response = requests.get(
        f"{api_base}/models", 
        headers={"Authorization": f"Bearer {api_key}"},
        timeout=10
    )
    
    if response.status_code == 200:
        data = response.json()
        models = data.get('data', [])
        
        targets = ['llama', 'mistral', 'gemma', 'gpt']
        found_models = []
        
        for m in models:
            mid = m['id']
            # Αν το ID περιέχει κάποια από τις λέξεις κλειδιά
            if any(t in mid.lower() for t in targets):
                found_models.append(mid)
        
        print(json.dumps(found_models, indent=4))
        
    else:
        print(f"Error: {response.text}")

except Exception as e:
    print(f"Connection Error: {e}")

[
    "mistral-large-24.02",
    "gpt-oss-120b",
    "mistral-small-24.02",
    "mistral-7b-instruct-v0.2",
    "llama-3.2-1b",
    "gemma3-27b-it",
    "llama-3.1-8b",
    "llama-3.3-70b",
    "gemma3-27b-it-long",
    "llama-3.2-3b",
    "gpt-4o-mini",
    "gpt-4o",
    "llama-3.1-70b",
    "gpt-oss-20b"
]


**ΠΑΡΑΤΗΡΗΣΕΙΣ**

○ Τα υπόλοιπα μοντέλα (πχ llama-3.1-8b, mistral-7b-instruct-v0.2) σκάνε με Bedrock error όταν τα τρέχω

***ΟΛΑ ΤΑ ΜΑΘΗΜΑΤΑ, all closed-type questions*** (RQ1: Comparative Performance Across Subjects)

○ **Στο σύνολο των δεδομένων**, το gemma έχει accuracy 61% ενώ το krikri είχε 39%.

○ Στα επιμέρους μαθήματα παρατηρούνται τα εξής:

| Subject | Total Questions | gemma | krikri |
| :--- | :---: | :---: | :---: |
| ΓΛΩΣΣΑ | 723 | 71.6% | 45.2% |
| ΘΡΗΣΚΕΥΤΙΚΑ | 90 | 77.8% | 64.4% |
| ΜΑΘΗΜΑΤΙΚΑ | 555 | 44.3% | 26.3% |

***Multimodality VS Κείμενο*** (RQ2)

○ **Στο σύνολο των δεδομένων** παρατηρούνται τα εξής:

| Multimodality | Total Questions | gemma | krikri |
| :--- | :---: | :---: | :---: |
| no | 1176 | 63.5% | 40.6% |
| yes | 192 | 45.3% | 28.1% |

○ Για το gemma έχουμε πτώση 18.2% με την παρουσία εικόνας, ενώ για το krikri έχουμε πτώση μόλις 12.5%

***Αll types of closed questions (multiple-choice, fill-in-the-gaps, true/false, matching)*** (RQ3: Performance Across Exercise Types)

○ Ο τωρινός κώδικας (task_config) είναι φτιαγμένος να ψάχνει για ένα index, δηλαδή έναν αριθμό (π.χ. 1, 2, 3). Αυτό δουλεύει στο Multiple Choice, στο True/False και στο Fill-in-the-gaps γιατί εκεί η σωστή απάντηση είναι "Επιλογή 1" ή "Επιλογή 2". Στο Matching, η απάντηση δεν είναι ένας αριθμός. Είναι μια λίστα με ζεύγη (["1-γ", "2-α", "3-ε", "4-β", "5-δ"]). Στο dataset, το πεδίο answer_index είναι κενό (None) γιατί δεν υπάρχει "μία σωστή επιλογή", αλλά ένας συνδυασμός. Καλό θα ήταν να εξαιρέσουμε το Matching από αυτό το πείραμα (Closed Types), καθώς χρειάζεται άλλο κώδικα αξιολόγησης και πρόκειται για ελάχιστα παραδείγματα.

○ **Στο σύνολο των δεδομένων**, στα διαφορετικά είδη ερωτήσεων κλειστού τύπου παρατηρούνται τα εξής:

| Exercise Type    | Total Exercises | gemma | krikri |
| :--- | :---: | :---: | :---: |
| Fill-in-the-gaps | 9               | 88.9% | 77.8%  |
| Matching         | 4               | 0.0%  | 0.0%   |
| Multiple Choice  | 1268            | 59.3% | 37.1%  |
| True/False       | 87              | 85.1% | 60.9%  |

Πρόβλημα με matching ερωτήσεις

In [ ]:
# Έλεγχος: Δείξε μου μια Matching ερώτηση όπως είναι ΤΩΡΑ στο df
matching_check = df[df['Exercise Type'] == 'Matching'].head(1)

if not matching_check.empty:
    print("✅ Βρέθηκε Matching ερώτηση.")
    print("\n--- Choices (Επιλογές που βλέπει το μοντέλο) ---")
    # Τυπώνουμε την πρώτη εγγραφή της στήλης Choices
    print(matching_check['Choices'].iloc[0]) 
    
    print("\n--- Ground Truth (Σωστή απάντηση) ---")
    print(matching_check['Ground Truth'].iloc[0])
    
    print("\n--- Τι απάντησαν τα μοντέλα ---")
    for model in models_to_test:
        col = f"{model}_pred"
        if col in matching_check.columns:
            print(f"{model}: {matching_check[col].iloc[0]}")
else:
    print("Δεν βρέθηκαν Matching ερωτήσεις στο df")

✅ Βρέθηκε Matching ερώτηση.

--- Choices (Επιλογές που βλέπει το μοντέλο) ---
Α. θέατρο | Β. χορός | Γ. πανηγύρια | α. συναναστροφή | β. λατρεία | γ. διασκέδαση | δ. εύθυμη διάθεση | ε. καλή σωματική κατάσταση | στ. καλλιέργεια πνεύματος

--- Ground Truth (Σωστή απάντηση) ---
nan

--- Τι απάντησαν τα μοντέλα ---
gemma3-27b-it: 0
krikri-dpo-context: 0
